Bước 1.1: Đọc dữ liệu

In [1]:
import pandas as pd
import numpy as np

In [2]:
movies = pd.read_csv("../data/raw/movies_metadata.csv", low_memory=False)
credits = pd.read_csv("../data/raw/credits.csv")
keywords = pd.read_csv("../data/raw/keywords.csv")
links_small = pd.read_csv("../data/raw/links_small.csv")
ratings_small = pd.read_csv("../data/raw/ratings_small.csv")

In [3]:
print("movies:", movies.shape)
print("credits:", credits.shape)
print("keywords:", keywords.shape)
print("links_small:", links_small.shape)
print("ratings_small:", ratings_small.shape)

movies: (45466, 24)
credits: (45476, 3)
keywords: (46419, 2)
links_small: (9125, 3)
ratings_small: (100004, 4)


In [4]:
print(movies.info())
print("-"*50)
print(credits.info())
print("-"*50)
print(keywords.info())

<class 'pandas.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  str    
 1   belongs_to_collection  4494 non-null   str    
 2   budget                 45466 non-null  str    
 3   genres                 45466 non-null  str    
 4   homepage               7782 non-null   str    
 5   id                     45466 non-null  str    
 6   imdb_id                45449 non-null  str    
 7   original_language      45455 non-null  str    
 8   original_title         45466 non-null  str    
 9   overview               44512 non-null  str    
 10  popularity             45461 non-null  str    
 11  poster_path            45080 non-null  str    
 12  production_companies   45463 non-null  str    
 13  production_countries   45463 non-null  str    
 14  release_date           45379 non-null  str    
 15  revenue      

In [5]:
movies.head(3)

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0


In [6]:
movies[['id', 'title', 'genres', 'overview']].iloc[0]

id                                                        862
title                                               Toy Story
genres      [{'id': 16, 'name': 'Animation'}, {'id': 35, '...
overview    Led by Woody, Andy's toys live happily in his ...
Name: 0, dtype: str

In [7]:
credits.head(2)

,cast,crew,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",8844


In [8]:
keywords.head(2)

,id,keywords
0,862,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
1,8844,"[{'id': 10090, 'name': 'board game'}, {'id': 1..."


Bước 1.2: Kiểm tra id lỗi + trùng lặp

In [9]:
# Kiểm tra các dòng có id không phải số trong movies
non_numeric_ids = movies[~movies['id'].astype(str).str.isnumeric()]
print("Số dòng id không hợp lệ:", non_numeric_ids.shape[0])
non_numeric_ids[['id', 'title']]

Số dòng id không hợp lệ: 3


,id,title
19730,1997-08-20,NaN
29503,2012-09-29,NaN
35587,2014-01-01,NaN


In [10]:
# Kiểm tra trùng lặp id trong credits và keywords
print("Credits - id trùng lặp:", credits['id'].duplicated().sum())
print("Keywords - id trùng lặp:", keywords['id'].duplicated().sum())
print("Movies - id trùng lặp (sau khi loại id lỗi):", 
      movies[movies['id'].astype(str).str.isnumeric()]['id'].astype('int64').duplicated().sum())

Credits - id trùng lặp: 44
Keywords - id trùng lặp: 987
Movies - id trùng lặp (sau khi loại id lỗi): 30


In [11]:
# Kiểm tra trùng lặp toàn dòng và theo title
print("Số dòng trùng lặp hoàn toàn:", movies.duplicated().sum())
print("Số title trùng lặp:", movies['title'].duplicated().sum())

Số dòng trùng lặp hoàn toàn: 17
Số title trùng lặp: 3188


In [12]:
# Kiểm tra ratings_small và links_small
print(ratings_small.info())
print(ratings_small.head())
print("-"*50)
print(links_small.info())
print(links_small.head())

<class 'pandas.DataFrame'>
RangeIndex: 100004 entries, 0 to 100003
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100004 non-null  int64  
 1   movieId    100004 non-null  int64  
 2   rating     100004 non-null  float64
 3   timestamp  100004 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB
None
   userId  movieId  rating   timestamp
0       1       31     2.5  1260759144
1       1     1029     3.0  1260759179
2       1     1061     3.0  1260759182
3       1     1129     2.0  1260759185
4       1     1172     4.0  1260759205
--------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 9125 entries, 0 to 9124
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  9125 non-null   int64  
 1   imdbId   9125 non-null   int64  
 2   tmdbId   9112 non-null   float64
dtypes: float64(1), int64(2)
m

Bước 1.3: Xử lý trùng lặp + merge dữ liệu

In [13]:
# Bổ sung: bảng đầy đủ giá trị thiếu theo cột
movies.isnull().sum().sort_values(ascending=False)

belongs_to_collection    40972
homepage                 37684
tagline                  25054
overview                   954
poster_path                386
runtime                    263
status                      87
release_date                87
imdb_id                     17
original_language           11
vote_average                 6
vote_count                   6
title                        6
video                        6
spoken_languages             6
revenue                      6
popularity                   5
production_countries         3
production_companies         3
genres                       0
id                           0
adult                        0
budget                       0
original_title               0
dtype: int64

In [14]:
# Loại bỏ 3 dòng id lỗi, ép kiểu id về int64
movies_clean = movies[movies['id'].astype(str).str.isnumeric()].copy()
movies_clean['id'] = movies_clean['id'].astype('int64')

# Loại bỏ trùng lặp toàn dòng và trùng id (giữ bản đầu tiên)
movies_clean = movies_clean.drop_duplicates()
movies_clean = movies_clean.drop_duplicates(subset='id', keep='first')

# Loại bỏ id trùng ở credits, keywords
credits_clean = credits.drop_duplicates(subset='id', keep='first')
keywords_clean = keywords.drop_duplicates(subset='id', keep='first')

print("movies_clean:", movies_clean.shape)
print("credits_clean:", credits_clean.shape)
print("keywords_clean:", keywords_clean.shape)

movies_clean: (45433, 24)
credits_clean: (45432, 3)
keywords_clean: (45432, 2)


In [15]:
# Merge movies + credits + keywords theo id
df = movies_clean.merge(credits_clean, on='id', how='left')
df = df.merge(keywords_clean, on='id', how='left')

print("Sau khi merge:", df.shape)
df.isnull().sum()[['cast', 'crew', 'keywords']] if 'cast' in df.columns else print("kiểm tra tên cột")

Sau khi merge: (45433, 27)


cast        1
crew        1
keywords    1
dtype: int64

In [16]:
# Kiểm tra vài dòng sau merge để chắc chắn khớp đúng
df[['id', 'title', 'genres', 'cast', 'crew', 'keywords']].head(3)

,id,title,genres,cast,crew,keywords
0,862,Toy Story,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...","[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
1,8844,Jumanji,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...","[{'id': 10090, 'name': 'board game'}, {'id': 1..."
2,15602,Grumpier Old Men,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...","[{'id': 1495, 'name': 'fishing'}, {'id': 12392..."


Bước 1.4: Xử lý các dòng thiếu quan trọng

In [17]:
# Loại 6 dòng thiếu title (thường đi kèm thiếu nhiều cột khác luôn)
df = df.dropna(subset=['title'])

# Loại 1 dòng thiếu cast/crew/keywords sau merge
df = df.dropna(subset=['cast', 'crew', 'keywords'])

# Điền rỗng cho overview thiếu (giữ dòng lại)
df['overview'] = df['overview'].fillna('')

print("Shape sau khi xử lý thiếu:", df.shape)

Shape sau khi xử lý thiếu: (45429, 27)


In [18]:
# Chọn các cột cần thiết, bỏ cột không dùng
keep_cols = ['id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew',
             'popularity', 'vote_average', 'vote_count', 'release_date', 'poster_path']

df_final = df[keep_cols].copy()
print(df_final.shape)
df_final.isnull().sum()

(45429, 12)


id                0
title             0
overview          0
genres            0
keywords          0
cast              0
crew              0
popularity        0
vote_average      0
vote_count        0
release_date     84
poster_path     383
dtype: int64

In [19]:
# Kiểm tra kiểu dữ liệu các cột số trước khi ép kiểu ở bước sau
df_final.dtypes

id                int64
title               str
overview            str
genres              str
keywords            str
cast                str
crew                str
popularity          str
vote_average    float64
vote_count      float64
release_date        str
poster_path         str
dtype: object

In [20]:
import os
os.makedirs('../data/interim', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

df_final.to_csv('../data/interim/movies_merged_basic.csv', index=False)
print("Đã lưu movies_merged_basic.csv")

Đã lưu movies_merged_basic.csv


In [21]:
# Lưu tạm kết quả bước cleaning cơ bản (chưa parse JSON) ra interim
df_final.to_csv('../data/interim/movies_merged_basic.csv', index=False)
print("Đã lưu movies_merged_basic.csv")

Đã lưu movies_merged_basic.csv


Bước 1.5: Parse JSON-string

In [22]:
import ast

# Hàm parse chung: lấy list các 'name' từ JSON-string
def parse_names(text, top_n=None):
    try:
        items = ast.literal_eval(text)
        names = [i['name'] for i in items]
        return names[:top_n] if top_n else names
    except (ValueError, SyntaxError):
        return []

# Hàm lấy đạo diễn từ crew
def parse_director(text):
    try:
        items = ast.literal_eval(text)
        for i in items:
            if i.get('job') == 'Director':
                return i['name']
        return ''
    except (ValueError, SyntaxError):
        return ''

In [23]:
# Áp dụng parse cho từng cột
df_final['genres_list'] = df_final['genres'].apply(lambda x: parse_names(x))
df_final['keywords_list'] = df_final['keywords'].apply(lambda x: parse_names(x))
df_final['cast_list'] = df_final['cast'].apply(lambda x: parse_names(x, top_n=3))
df_final['director'] = df_final['crew'].apply(parse_director)

# Kiểm tra kết quả trên vài dòng
df_final[['title', 'genres_list', 'keywords_list', 'cast_list', 'director']].head(5)

,title,genres_list,keywords_list,cast_list,director
0,Toy Story,"[Animation, Comedy, Family]","[jealousy, toy, boy, friendship, friends, riva...","[Tom Hanks, Tim Allen, Don Rickles]",John Lasseter
1,Jumanji,"[Adventure, Fantasy, Family]","[board game, disappearance, based on children'...","[Robin Williams, Jonathan Hyde, Kirsten Dunst]",Joe Johnston
2,Grumpier Old Men,"[Romance, Comedy]","[fishing, best friend, duringcreditsstinger, o...","[Walter Matthau, Jack Lemmon, Ann-Margret]",Howard Deutch
3,Waiting to Exhale,"[Comedy, Drama, Romance]","[based on novel, interracial relationship, sin...","[Whitney Houston, Angela Bassett, Loretta Devine]",Forest Whitaker
4,Father of the Bride Part II,[Comedy],"[baby, midlife crisis, confidence, aging, daug...","[Steve Martin, Diane Keaton, Martin Short]",Charles Shyer


In [24]:
# Kiểm tra xem có bao nhiêu phim bị rỗng sau khi parse (dấu hiệu lỗi định dạng)
print("genres rỗng:", (df_final['genres_list'].str.len() == 0).sum())
print("keywords rỗng:", (df_final['keywords_list'].str.len() == 0).sum())
print("cast rỗng:", (df_final['cast_list'].str.len() == 0).sum())
print("director rỗng:", (df_final['director'] == '').sum())

genres rỗng: 2441
keywords rỗng: 14340
cast rỗng: 2414
director rỗng: 887


In [25]:
# Ép kiểu số cho popularity
df_final['popularity'] = pd.to_numeric(df_final['popularity'], errors='coerce')
print("popularity lỗi ép kiểu (NaN mới sinh ra):", df_final['popularity'].isnull().sum())

# Ép kiểu ngày tháng cho release_date, tách năm
df_final['release_date'] = pd.to_datetime(df_final['release_date'], errors='coerce')
df_final['release_year'] = df_final['release_date'].dt.year
print("release_date lỗi ép kiểu:", df_final['release_date'].isnull().sum())

popularity lỗi ép kiểu (NaN mới sinh ra): 0
release_date lỗi ép kiểu: 84


In [26]:
df_final.dtypes

id                        int64
title                       str
overview                    str
genres                      str
keywords                    str
cast                        str
crew                        str
popularity              float64
vote_average            float64
vote_count              float64
release_date     datetime64[us]
poster_path                 str
genres_list              object
keywords_list            object
cast_list                object
director                    str
release_year            float64
dtype: object

Bước 1.6: Chuẩn hóa chuỗi 

In [27]:
# Chuẩn hóa: nối tên riêng thành 1 từ (tránh "Tom Hanks" và "Tom Cruise" bị chung token "Hanks"/"Cruise"),
# chuyển về chữ thường, bỏ khoảng trắng thừa
def clean_token_list(lst):
    return [str(x).replace(' ', '').strip().lower() for x in lst]

df_final['genres_clean'] = df_final['genres_list'].apply(clean_token_list)
df_final['keywords_clean'] = df_final['keywords_list'].apply(clean_token_list)
df_final['cast_clean'] = df_final['cast_list'].apply(clean_token_list)
df_final['director_clean'] = df_final['director'].apply(lambda x: str(x).replace(' ', '').strip().lower())

df_final[['title', 'genres_clean', 'keywords_clean', 'cast_clean', 'director_clean']].head(5)

,title,genres_clean,keywords_clean,cast_clean,director_clean
0,Toy Story,"[animation, comedy, family]","[jealousy, toy, boy, friendship, friends, riva...","[tomhanks, timallen, donrickles]",johnlasseter
1,Jumanji,"[adventure, fantasy, family]","[boardgame, disappearance, basedonchildren'sbo...","[robinwilliams, jonathanhyde, kirstendunst]",joejohnston
2,Grumpier Old Men,"[romance, comedy]","[fishing, bestfriend, duringcreditsstinger, ol...","[waltermatthau, jacklemmon, ann-margret]",howarddeutch
3,Waiting to Exhale,"[comedy, drama, romance]","[basedonnovel, interracialrelationship, single...","[whitneyhouston, angelabassett, lorettadevine]",forestwhitaker
4,Father of the Bride Part II,[comedy],"[baby, midlifecrisis, confidence, aging, daugh...","[stevemartin, dianekeaton, martinshort]",charlesshyer


In [28]:
# Kiểm tra lại toàn bộ giá trị thiếu lần cuối trước khi lưu
df_final.isnull().sum()

id                  0
title               0
overview            0
genres              0
keywords            0
cast                0
crew                0
popularity          0
vote_average        0
vote_count          0
release_date       84
poster_path       383
genres_list         0
keywords_list       0
cast_list           0
director            0
release_year       84
genres_clean        0
keywords_clean      0
cast_clean          0
director_clean      0
dtype: int64

In [29]:
# Lưu file cleaning hoàn chỉnh (chưa vector hóa) ra processed
df_final.to_csv('../data/processed/movies_clean.csv', index=False)
print("Đã lưu movies_clean.csv, shape:", df_final.shape)

Đã lưu movies_clean.csv, shape: (45429, 21)


In [30]:
# Xử lý sơ bộ ratings_small + links_small, lưu ra processed luôn để dùng cho bước CF sau này
ratings_clean = ratings_small.drop_duplicates()
print("ratings_clean:", ratings_clean.shape)

ratings_clean.to_csv('../data/processed/ratings_clean.csv', index=False)
print("Đã lưu ratings_clean.csv")

ratings_clean: (100004, 4)
Đã lưu ratings_clean.csv
